In [1]:
import os
if os.getenv("CUDA_VISIBLE_DEVICES") is None:
    gpu_num = 0 # Use "" to use the CPU
    os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
# tahtan
# Import Sionna
import sys
sys.path.append('../')
import sionna

# try:
#     import sionna
# except ImportError as e:
#     # Install Sionna if package is not already installed
#     import os
#     os.system("pip install sionna")
#     import sionna

import tensorflow as tf
# Configure the notebook to use only a single GPU and allocate only as much memory as needed
# For more details, see https://www.tensorflow.org/guide/gpu
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        print(e)
# Avoid warnings from TensorFlow
tf.get_logger().setLevel('ERROR')

sionna.config.seed = 42 # Set seed for reproducible results

# Load the required Sionna components
from sionna.nr import PUSCHConfig, PUSCHTransmitter, PUSCHReceiver, CarrierConfig, PUSCHDMRSConfig,\
                        TBConfig, PUSCHPilotPattern, TBEncoder, PUSCHPrecoder, LayerMapper, LayerDemapper, check_pusch_configs,\
                        TBDecoder, PUSCHLSChannelEstimator
from sionna.nr.utils import generate_prng_seq
from sionna.channel import AWGN, RayleighBlockFading, OFDMChannel, TimeChannel, time_lag_discrete_time_channel
from sionna.channel.utils import * 
from sionna.channel.tr38901 import Antenna, AntennaArray, UMi, UMa, RMa, TDL, CDL
from sionna.channel import gen_single_sector_topology as gen_topology
from sionna.utils import compute_ber, ebnodb2no, sim_ber, array_to_hash, create_timestamped_folders, b2b, f2f, BinarySource
from sionna.ofdm import KBestDetector, LinearDetector, MaximumLikelihoodDetector,\
        LSChannelEstimator, LMMSEEqualizer, RemoveNulledSubcarriers, ResourceGridDemapper,\
        ResourceGrid, ResourceGridMapper, OFDMModulator
from sionna.mimo import StreamManagement
from sionna.mapping import Mapper, Demapper


In [2]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time
from datetime import datetime, timedelta
# from bs4 import BeautifulSoup
import pickle
from collections import namedtuple
import json
from tqdm.notebook import tqdm
import itertools
import io
import h5py

## A Hello World Example

Let us start with a simple "Hello, World!" example in which we will simulate PUSCH transmissions from a single transmitter to a single receiver over an AWGN channel.

In [3]:
# from dataclasses import dataclass, field
# from typing import List

# @dataclass
# class SystemConfig:
#     NCellId: int = 246
#     FrequencyRange: int = 1
#     BandWidth: int = 100
#     Numerology: int = 1
#     CpType: int = 0
#     NTxAnt: int = 1
#     NRxAnt: int = 8
#     BwpNRb: int = 273
#     BwpRbOffset: int = 0
#     harqProcFlag: int = 0
#     nHarqProc: int = 1
#     rvSeq: int = 0


# @dataclass
# class UeConfig:
#     TransformPrecoding: int = 0
#     Rnti: int = 20002
#     nId: int = 246
#     CodeBookBased: int = 0
#     DmrsPortSetIdx: List[int] = field(default_factory=lambda: [0])  # FIXED
#     NLayers: int = 1
#     NumDmrsCdmGroupsWithoutData: int = 2
#     Tpmi: int = 0
#     FirstSymb: int = 0
#     NPuschSymbAll: int = 14
#     RaType: int = 1
#     FirstPrb: int = 31
#     NPrb: int = 4
#     FrequencyHoppingMode: int = 0
#     McsTable: int = 0
#     Mcs: int = 3
#     ILbrm: int = 0
#     nScId: int = 0
#     NnScIdId: int = 246
#     DmrsConfigurationType: int = 0
#     DmrsDuration: int = 1
#     DmrsAdditionalPosition: int = 1
#     PuschMappingType: int = 0
#     DmrsTypeAPosition: int = 3
#     HoppingMode: int = 0
#     NRsId: int = 0
#     Ptrs: int = 0
#     ScalingFactor: int = 0
#     OAck: int = 0
#     IHarqAckOffset: int = 11
#     OCsi1: int = 0
#     ICsi1Offset: int = 7
#     OCsi2: int = 0
#     ICsi2Offset: int = 0
#     NPrbOh: int = 0
#     nCw: int = 1
#     TpPi2Bpsk: int = 0

# @dataclass
# class MyConfig:
#     Sys: SystemConfig
#     Ue: List[UeConfig]
#     Num_tx: int = 1
#     Num_rx: int = 1
#     Carrier_frequency: float = 2.55e9  # Carrier frequency in Hz

# # Example usage
# My_Config = MyConfig(SystemConfig(), [UeConfig()])


In [4]:
# class MyPUSCHConfig(PUSCHConfig):
#     def __init__(self, My_Config: MyConfig):
#         self.My_Config = My_Config
#         super().__init__(
#             carrier_config=CarrierConfig(
#                 n_cell_id=My_Config.Sys.NCellId,
#                 cyclic_prefix="normal" if ~My_Config.Sys.CpType else "extended",
#                 subcarrier_spacing=15*(2**My_Config.Sys.Numerology),
#                 n_size_grid=My_Config.Sys.BwpNRb,
#                 n_start_grid=My_Config.Sys.BwpRbOffset,
#                 slot_number=4,
#                 frame_number=0
#             ),
#             pusch_dmrs_config=PUSCHDMRSConfig(
#                 config_type=My_Config.Ue[0].DmrsConfigurationType + 1,
#                 length=My_Config.Ue[0].DmrsDuration,
#                 additional_position=My_Config.Ue[0].DmrsAdditionalPosition,
#                 dmrs_port_set=My_Config.Ue[0].DmrsPortSetIdx,
#                 n_id=My_Config.Ue[0].NnScIdId,
#                 n_scid=My_Config.Ue[0].nScId,
#                 num_cdm_groups_without_data=My_Config.Ue[0].NumDmrsCdmGroupsWithoutData,
#                 type_a_position=My_Config.Ue[0].DmrsTypeAPosition
#             ),
#             tb_config=TBConfig(
#                 channel_type='PUSCH',
#                 n_id=My_Config.Ue[0].nId,
#                 mcs_table=My_Config.Ue[0].McsTable + 1,
#                 mcs_index=My_Config.Ue[0].Mcs
#             ),
#             mapping_type='A' if ~My_Config.Ue[0].PuschMappingType else 'B',
#             n_size_bwp=My_Config.Sys.BwpNRb,
#             n_start_bwp=My_Config.Sys.BwpRbOffset,
#             num_layers=My_Config.Ue[0].NLayers,
#             num_antenna_ports=len(My_Config.Ue[0].DmrsPortSetIdx),
#             precoding='non-codebook' if ~My_Config.Ue[0].CodeBookBased else 'codebook',
#             tpmi=My_Config.Ue[0].Tpmi,
#             transform_precoding=False if ~My_Config.Ue[0].TransformPrecoding else True,
#             n_rnti=My_Config.Ue[0].Rnti,
#             symbol_allocation=[My_Config.Ue[0].FirstSymb,My_Config.Ue[0].NPuschSymbAll]
#         )
#     @property
#     def first_resource_block(self):
#         """
#         :class:`~sionna.nr.CarrierConfig` : Carrier configuration
#         """
#         return self.My_Config.Ue[0].FirstPrb
    
#     @property
#     def first_subcarrier(self):
#         """
#         :class:`~sionna.nr.CarrierConfig` : Carrier configuration
#         """
#         return 12*self.first_resource_block
    
#     @property
#     def num_resource_blocks(self):
#         """
#         int, read-only : Number of allocated resource blocks for the
#             PUSCH transmissions.
#         """
#         return self.My_Config.Ue[0].NPrb

#     @property
#     def dmrs_grid(self):
#         # pylint: disable=line-too-long
#         """
#         complex, [num_dmrs_ports, num_subcarriers, num_symbols_per_slot], read-only : Empty
#             resource grid for each DMRS port, filled with DMRS signals

#             This property returns for each configured DMRS port an empty
#             resource grid filled with DMRS signals as defined in
#             Section 6.4.1.1 [3GPP38211]. Not all possible options are implemented,
#             e.g., frequency hopping and transform precoding are not available.

#             This property provides the *unprecoded* DMRS for each configured DMRS port.
#             Precoding might be applied to map the DMRS to the antenna ports. However,
#             in this case, the number of DMRS ports cannot be larger than the number of
#             layers.
#         """
#         # Check configuration
#         self.check_config()

#         # Configure DMRS ports set if it has not been set
#         reset_dmrs_port_set = False
#         if len(self.dmrs.dmrs_port_set)==0:
#             self.dmrs.dmrs_port_set = list(range(self.num_layers))
#             reset_dmrs_port_set = True

#         # Generate empty resource grid for each port
#         a_tilde = np.zeros([len(self.dmrs.dmrs_port_set),
#                             self.num_subcarriers,
#                             self.carrier.num_symbols_per_slot],
#                             dtype=complex)
#         first_subcarrier = self.first_subcarrier
#         num_subcarriers = self.num_subcarriers

#         # For every l_bar
#         for l_bar in self.l_bar:

#             # For every l_prime
#             for l_prime in self.l_prime:

#                 # Compute c_init
#                 l = l_bar + l_prime
#                 c_init = self.c_init(l)
#                 # Generate RNG
#                 c = generate_prng_seq(first_subcarrier + num_subcarriers, c_init=c_init)
#                 c = c[first_subcarrier:]

#                 # Map to QAM
#                 r = 1/np.sqrt(2)*((1-2*c[::2]) + 1j*(1-2*c[1::2]))

#                 # For every port in the dmrs port set
#                 for j_ind, _ in enumerate(self.dmrs.dmrs_port_set):

#                     # For every n
#                     for n in self.n:

#                         # For every k_prime
#                         for k_prime in [0, 1]:

#                             if self.dmrs.config_type==1:
#                                 k = 4*n + 2*k_prime + \
#                                     self.dmrs.deltas[j_ind]
#                             else: # config_type == 2
#                                 k = 6*n + k_prime + \
#                                     self.dmrs.deltas[j_ind]

#                             a_tilde[j_ind, k, self.l_ref+l] = \
#                                 r[2*n + k_prime] * \
#                                 self.dmrs.w_f[k_prime][j_ind] * \
#                                 self.dmrs.w_t[l_prime][j_ind]

#         # Amplitude scaling
#         a = self.dmrs.beta*a_tilde

#         # Reset DMRS port set if it was not set
#         if reset_dmrs_port_set:
#             self.dmrs.dmrs_port_set = []

#         return a
    
# Pusch_Config = MyPUSCHConfig(My_Config)
# Pusch_Config.show()

In [5]:
def load_pickle(filename):
    """Saves data to a pickle file."""
    with open(filename, "rb") as f:
        return pickle.load(f)

# Function to read a single sample from HDF5
def load_hdf5(filename, name):
    with h5py.File(filename, "r") as f:
        grp = f[f"{name}"]
        b = grp["b"][:]
        c = grp["c"][:]
        y = grp["y"][:]
    return b, c, y

def data_loader(df, dir, from_pickle=False):
    #  # .sample(frac=1) for shuffing
    for pusch_record in df.sample(frac=1).itertuples():
        data_filename = pusch_record.Data_filename
        data_dirname = pusch_record.Data_dirname
        esno_db = pusch_record.Esno_db
        index = pusch_record.Index
        # 1 tx
        if from_pickle:
            b = load_pickle(f'{dir}/{data_dirname}/{data_filename}.b.pkl')
            c = load_pickle(f'{dir}/{data_dirname}/{data_filename}.c.pkl')
            y = load_pickle(f'{dir}/{data_dirname}/{data_filename}.y.pkl')
        else:
            b, c, y = load_hdf5(f'{dir}/{data_dirname}.hdf5', f'{data_filename}')

        c_len = tf.shape(c)[-1]
        b_len = tf.shape(b)[-1]
        b = tf.pad(b, [[0,c_len-b_len]])  # Pad b with zeros to match c
        
        yield index, esno_db, c, y, b, b_len

def preprocessing(index, esno_db, c, y, b, b_len):
    c = tf.transpose(tf.reshape(c, [12,-1,2]), perm=[2,0,1]) # 2 dmrs
    y = tf.concat([tf.math.real(y), tf.math.imag(y)], axis=0)

    return index, esno_db, c, y, b, b_len

dataset_dir = f'../Pusch_data/dataset'
pickles_dir = f'{dataset_dir}/pickle'
hdf5_dir = f'{dataset_dir}/hdf5'
parquet_path = f'{dataset_dir}/parquet/20250304031450018139.parquet'
df = pd.read_parquet(parquet_path, engine="pyarrow")
# df = df[(df['nMCS'] == 9) & (df['nSlot'] == 4)] 

dataset = tf.data.Dataset.from_generator(
            lambda: data_loader(df, hdf5_dir),
            output_types=(tf.int32, tf.float32, tf.float32, tf.complex64, tf.float32, tf.int32))

In [6]:
for n, (index, esno_db, c, y, b, b_len) in enumerate(dataset.map(preprocessing).batch(32)):
    print(index, n, esno_db.shape, c.shape, y.shape, b.shape, b_len.shape)

tf.Tensor([ 5  8  0  1 11  9  2  6  7 10  3  4], shape=(12,), dtype=int32) 0 (12,) (12, 2, 12, 48) (12, 16, 14, 48) (12, 1152) (12,)
